In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from benchmark_data_leakage.utils import find_repo_root

repo_root = find_repo_root()
data_dir = repo_root / "data"
out_dir = data_dir / "out"
fig_dir = repo_root / "figures"
fig_dir.mkdir(exist_ok=True)

In [ ]:
FEPp_data_points = [
    # These are from the Boltz-2 and IsoDDE papers
    ("IsoDDE", 0.85),
    ("FEP+", 0.78),
    ("OpenFE", 0.66),
    ("Boltz-2", 0.66),
    # These were extracted with a ligand only model, and can be reproduced by following:
    # https://github.com/bamattsson/timesplit-affinity-benchmark/blob/888d02930879da1287f02b7d18d4bc5cc8e4d274/reproduce_results_on_fep4/00_instructions.md
    # Modify 03_train.yaml to vary which features are used
    ("FP + Mol Prop", 0.663),
    ("FP only", 0.529),
    ("Mol Prop only", 0.486)
]

In [ ]:
C_PHYSICS  = "#93cfe0"   # pale teal
C_AI_MODEL = "#e8a8b0"   # pale rose
C_BASELINE = "#d8d898"   # pale olive

labels  = [m for m, _ in FEPp_data_points]
values  = [v for _, v in FEPp_data_points]
colours = [C_AI_MODEL, C_PHYSICS, C_PHYSICS, C_AI_MODEL, C_BASELINE, C_BASELINE, C_BASELINE]

fig, ax = plt.subplots(figsize=(5.0, 4.8))
x_pos = np.arange(len(labels)) * 0.8

bars = ax.bar(x_pos, values, color=colours, width=0.45, zorder=2,
              edgecolor="#323232", linewidth=0.6)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val / 2,
            f"{val:.2f}", ha="center", va="center", fontsize=9,
            rotation=90, color="black")

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9, rotation=90, ha="center")
ax.set_ylabel("Mean pearson correlation ($r$)", fontsize=11)
ax.set_ylim(0, 1.05)

ax.grid(visible=True, which="major", axis="y", linestyle=":", linewidth=0.5, color="#cccccc")
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", length=0, labelsize=9)
ax.tick_params(axis="y", labelsize=9)

legend_handles = [
    mpatches.Patch(color=C_PHYSICS,  label="Physics-based methods"),
    mpatches.Patch(color=C_AI_MODEL, label="Structure-based ML models"),
    mpatches.Patch(color=C_BASELINE, label="Ligand-only ML baselines"),
]
ax.legend(handles=legend_handles, fontsize=9, frameon=False, loc="upper right")

plt.tight_layout()
plt.savefig(fig_dir / "FEPp_w_baselines_barplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
data_set_comparison = [
    # dataset, model, average pearson_r, SE of average pearson_r
    # For reproducing, see note above
    # n_rows = 87
    ("FEP+ 4", "FP + Mol Prop", 0.663, 0.055),
    ("FEP+ 4", "FP only", 0.529, 0.098),
    ("FEP+ 4", "Mol Prop only", 0.486, 0.150),

    # For reproducing, use:
    # https://github.com/bamattsson/timesplit-affinity-benchmark/tree/8c8ba5765af69537492f701581537663151c41a8
    # and follow README.md (1->2) and docs/baseline.md (1->3) with the following changes
    # benchmark.yaml:
    #   pipeline.tanimoto_threshold: null for timesplit (w/o novelty)
    # train.yaml:
    #   model.use_*: true/false
    # Prediction command (step 3) should be with --n-bootstraps 1000

    # n_rows = 28,568, n_assays = 910
    ("timesplit (w/o novelty)", "FP + Mol Prop", 0.3050, 0.0108),
    ("timesplit (w/o novelty)", "FP only", 0.3164, 0.0111),
    ("timesplit (w/o novelty)", "Mol Prop only", 0.1701, 0.0113),

    # n_rows = 1,108, n_assays = 24
    ("test", "FP + Mol Prop", -0.0001, 0.0487),
    ("test", "FP only", 0.1238, 0.0723),
    ("test", "Mol Prop only", 0.1159, 0.0745),
    # ("test", "FP + Mol Prop", -0.0067, 0.0498),
    # ("test", "FP only", 0.1170, 0.0678),
    # ("test", "Mol Prop only", 0.1147, 0.0716),
]

In [ ]:
from matplotlib.legend_handler import HandlerBase
from matplotlib.patches import Rectangle as MplRectangle


C_FEPP4 = "#d8d898"
C_TIMESPLIT = "#e8a8b0"   # pale rose
C_TEST  = "#93cfe0"   # pale teal

dataset_colors = {
    "FEP+ 4": C_FEPP4,
    "timesplit (w/o novelty)": C_TIMESPLIT,
    "test": C_TEST,
}

datasets_in_order = ["FEP+ 4", "timesplit (w/o novelty)", "test"]
models_in_order = ["FP + Mol Prop", "FP only", "Mol Prop only"]

bar_width = 0.6
bar_spacing = 0.75  # center-to-center within a group
group_gap = 0.5     # extra gap between groups

# Build x positions
x_pos = []
current_x = 0.0
for dataset in datasets_in_order:
    for i in range(len(models_in_order)):
        x_pos.append(current_x)
        if i < len(models_in_order) - 1:
            current_x += bar_spacing
    current_x += bar_spacing + group_gap

labels        = [m for d, m, p, s in data_set_comparison]
values        = [p for d, m, p, s in data_set_comparison]
ses           = [s for d, m, p, s in data_set_comparison]
datasets_list = [d for d, m, p, s in data_set_comparison]
colours       = [dataset_colors[d] for d in datasets_list]

fig, ax = plt.subplots(figsize=(10.0, 4.8))

bars = ax.bar(
    x_pos, values, color=colours, width=bar_width, zorder=2,
    edgecolor="#323232", linewidth=0.6,
    yerr=ses, capsize=3,
    error_kw=dict(elinewidth=0.8, ecolor="#323232", capthick=0.8, zorder=5),
)

ylim_bottom = -0.15
ylim_top = 1.0
ax.set_ylim(ylim_bottom, ylim_top)

# Value labels: horizontal text, above (+ve SE cap) or below (-ve SE cap)
for bar, val, se in zip(bars, values, ses):
    x = bar.get_x() + bar.get_width() / 2
    if val >= 0:
        y, va = val + se + 0.04, "bottom"
    else:
        y, va = val - se - 0.04, "top"
    ax.text(x, y, f"{val:.2f}", ha="center", va=va, fontsize=8, color="black")

ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=9, rotation=90, ha="center")
ax.set_ylabel("Mean Pearson correlation ($r$)", fontsize=11)

ax.grid(visible=True, which="major", axis="y", linestyle=":", linewidth=0.5, color="#cccccc")
ax.set_axisbelow(True)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", length=0, labelsize=9)
ax.tick_params(axis="y", labelsize=9)

novel_handle = mpatches.Patch(label=r"Time-split (discarded compounds)")
legend_handles = [
    mpatches.Patch(color=C_FEPP4, edgecolor="#323232", linewidth=0.6, label=r"Protein split (FEP+ 4)"),
    mpatches.Patch(color=C_TIMESPLIT, edgecolor="#323232", linewidth=0.6, label=r"Time-split"),
    mpatches.Patch(color=C_TEST,  edgecolor="#323232", linewidth=0.6, label=r"Time-split + novelty filter"),
]
ax.legend(
    handles=legend_handles,
    fontsize=9, frameon=False, loc="upper right",
)

plt.tight_layout()
plt.savefig(fig_dir / "dataset_baseline_comparison.png", dpi=300, bbox_inches="tight")
plt.show()